# Disease ↔ Anatomy Relation-Wise Merge

Merges Disease–Anatomy triples from Monarch, DRKG, Hetionet, and TARKG;
resolves disease names via DO/MESH, anatomy names via UBERON;
deduplicates by `(head, relation, tail)`; and saves the result.

## 0. Configuration

In [81]:
import pandas as pd
import numpy as np

# ── Base directories ──────────────────────────────────────────────────────────
BASE_DIR   = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR   = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'

# ── Output path ───────────────────────────────────────────────────────────────
OUT_PATH   = BASE_DIR + 'processed_data_relation_wise_merge/generalised/DISEASE_ANATOMY/ALL_DISEASE_ANATOMY.csv'

# ── Required output schema ────────────────────────────────────────────────────
REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Lookup Dictionaries

### 1a. UBERON — Anatomy tail nodes

In [57]:
UBERON = pd.read_csv(DB_DIR + 'uberon/Uberon_ID_Name_non_dup.csv')
UBERON = UBERON[~UBERON['Name'].isna()]
UBERON_dict = dict(zip(UBERON['UBERON ID'], UBERON['Name']))
print(f"UBERON dict: {len(UBERON_dict):,} entries")

UBERON dict: 15,668 entries


### 1b. Disease — DO, MedGen, MESH

In [58]:
DO_All_File = pd.read_csv(DB_DIR + 'DO/All_DO_ref_files.csv')
DOID_disname_dict      = dict(zip(DO_All_File['id'],    DO_All_File['label']))
DOID_disAltID_dict     = dict(zip(DO_All_File['id'],    DO_All_File['xrefs']))
DOID_disname_DOID_dict = dict(zip(DO_All_File['label'], DO_All_File['id']))
print(f"DO dict: {len(DOID_disname_dict):,} entries")

DO dict: 11,804 entries


In [59]:
MedGen_CUID_Source_ID = pd.read_csv(DB_DIR + 'MESH/MeSH/MedGenIDMappings.txt', sep='\t')
MedGen_CUID_Source_ID = MedGen_CUID_Source_ID.drop_duplicates(subset=['source_id', 'source'], keep='first')
MedGen_MeshID_name_dict = dict(zip(MedGen_CUID_Source_ID['source_id'], MedGen_CUID_Source_ID['pref_name']))

MONDO_2_MESH = MedGen_CUID_Source_ID[MedGen_CUID_Source_ID['source_id'].str.contains('MONDO', na=False)]
MONDO_2_MESH_dict    = dict(zip(MONDO_2_MESH['source_id'],     MONDO_2_MESH['#CUI_or_CN_id']))
MONDOMESH_2_des_dict = dict(zip(MONDO_2_MESH['#CUI_or_CN_id'], MONDO_2_MESH['pref_name']))
print(f"MONDO→MESH: {len(MONDO_2_MESH_dict):,} entries")

MONDO→MESH: 22,703 entries


In [60]:
Mesh2DOID = pd.read_csv(DB_DIR + 'DO/HumanDiseaseOntology-main/DOreports/MESHinDO.tsv', sep='\t')
Mesh2DOID.rename(columns={'ID': 'id', 'LABEL': 'label'}, inplace=True)
Mesh2DOID['xrefs'] = Mesh2DOID['xrefs'].str.split(', ')
Mesh2DOID = Mesh2DOID.explode('xrefs')
Mesh2DOID_dict = dict(zip(Mesh2DOID['xrefs'], Mesh2DOID['id']))

MESH = pd.read_csv(DB_DIR + 'MESH/MESH_Combined.csv')
MESH_dict = dict(zip(MESH['ID'], MESH['Name']))

Mesh_supp = pd.read_csv(DB_DIR + 'MESH/Mesh_Supplementary_Info.csv')
Mesh_supp_dict = dict(zip(Mesh_supp['ID'], Mesh_supp['Name']))

print(f"MESH dict: {len(MESH_dict):,} | MESH supp: {len(Mesh_supp_dict):,}")

MESH dict: 38,520 | MESH supp: 324,150


## 2. Helper — Resolve Disease Head Node

In [61]:
def resolve_disease_head(df):
    """Map MONDO→MESH, derive head_detail_name, assign head_id_is."""
    df['head'] = df['head'].map(MONDO_2_MESH_dict).fillna(df['head'])
    df = df[~df['head'].astype(str).str.startswith('MONDO')].copy()
    df['head_detail_name'] = df['head'].apply(
        lambda x: DOID_disname_dict.get(x)
                  if isinstance(x, str) and x.startswith('DOID')
                  else (MESH_dict.get(x) or Mesh_supp_dict.get(x) or MONDOMESH_2_des_dict.get(x))
    )
    df['head_id_is'] = np.where(
        df['head'].isna(), None,
        np.where(df['head'].astype(str).str.startswith('DOID'), 'DOID', 'MESH')
    )
    return df

## 3. Load KG Sources

### 3a. Monarch

In [62]:
Monarch_Disease_Anatomy = pd.read_csv(PROC_DIR + 'Monarch/Monarch_final/Human/Disease_AnatomicalEntity_Monarch.csv')
Monarch_Disease_Anatomy.columns = Monarch_Disease_Anatomy.columns.str.lower()
Monarch_Disease_Anatomy['kg_source'] = 'MonarchKG'
Monarch_Disease_Anatomy = resolve_disease_head(Monarch_Disease_Anatomy)
Monarch_Disease_Anatomy['tail_detail_name'] = Monarch_Disease_Anatomy['tail'].map(UBERON_dict)

print(f"Monarch:  {len(Monarch_Disease_Anatomy):,} rows")
Monarch_Disease_Anatomy['kg_type'] = 'Generalised'
Monarch_Disease_Anatomy['kg_source'] = 'Monarch_KG'
Monarch_Disease_Anatomy['species'] = 'Homo sapiens'
Monarch_Disease_Anatomy.head(2)

Monarch:  999 rows


,head,tail,relation_type,relation_source,kgsource,head_name,head_type,head_id_is,head_taxon,head_taxon_label,...,head_type_clean,tail_type_clean,relation,species_species,head_do_name,kg_source,head_detail_name,tail_detail_name,kg_type,species
0,C0729734,UBERON:0000304,disease_has_location,infores:mondo,MonarchKG,tendon sheath disorder,Disease,MESH,NaN,NaN,...,Disease,AnatomicalEntity,Disease_AnatomicalEntity,NaN,MONDO:0024876,Monarch_KG,Tendon sheath disorder,tendon sheath,Generalised,Homo sapiens
1,C1263793,UBERON:0002411,disease_has_location,infores:mondo,MonarchKG,clitoris neoplasm,Disease,MESH,NaN,NaN,...,Disease,AnatomicalEntity,Disease_AnatomicalEntity,NaN,MONDO:0024877,Monarch_KG,Clitoris neoplasm,clitoris,Generalised,Homo sapiens


### 3b. DRKG

In [63]:
DRKG_Disease_Anatomy = pd.read_csv(PROC_DIR + 'DRKG/DRKG_Disease_Anatomy.csv')
DRKG_Disease_Anatomy['Head_ID_IS'] = np.where(DRKG_Disease_Anatomy['Head'].str.startswith('DOID'), 'DOID', 'MESH')

DRKG_Disease_Anatomy.columns = DRKG_Disease_Anatomy.columns.str.lower()
print(f"DRKG:     {len(DRKG_Disease_Anatomy):,} rows")

DRKG_Disease_Anatomy['kg_type'] = 'Generalised'
DRKG_Disease_Anatomy['kg_source'] = 'DRKG'
DRKG_Disease_Anatomy['species'] = 'Homo sapiens'

DRKG_Disease_Anatomy.head(2)

DRKG:     3,591 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,head_orignal,tail_orignal,kg_type,species
0,D015209,Disease_Anatomy,UBERON:0002110,Disease,Hetionet::DlA::Disease:Anatomy,Anatomy,DRKG,"Cholangitis, Sclerosing",gallbladder,MESH,UBERON_ID,Disease::MESH:D015209,Anatomy::UBERON:0002110,Generalised,Homo sapiens
1,D024821,Disease_Anatomy,UBERON:0001980,Disease,Hetionet::DlA::Disease:Anatomy,Anatomy,DRKG,Metabolic Syndrome,arteriole,MESH,UBERON_ID,Disease::MESH:D024821,Anatomy::UBERON:0001980,Generalised,Homo sapiens


### 3c. Hetionet

In [64]:
Hetionet_Disease_Anatomy = pd.read_csv(PROC_DIR + 'Hetionet/Disease_Anatomy_Hetionet.csv')
Hetionet_Disease_Anatomy.columns = Hetionet_Disease_Anatomy.columns.str.lower()
print(f"Hetionet: {len(Hetionet_Disease_Anatomy):,} rows")

Hetionet_Disease_Anatomy['kg_type'] = 'Generalised'
Hetionet_Disease_Anatomy['kg_source'] = 'Hetionet'
Hetionet_Disease_Anatomy['species'] = 'Homo sapiens'
Hetionet_Disease_Anatomy.rename(columns={'head_name': 'head_detail_name', 'tail_name': 'tail_detail_name'}, inplace=True)


Hetionet_Disease_Anatomy.head(2)

Hetionet: 3,602 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,DOID:14268,Disease_Anatomy,UBERON:0002110,Disease,Disease–localizes–Anatomy,Anatomy,Hetionet,DOID,UBERON,sclerosing cholangitis,gallbladder,Generalised,Homo sapiens
1,DOID:14221,Disease_Anatomy,UBERON:0001980,Disease,Disease–localizes–Anatomy,Anatomy,Hetionet,DOID,UBERON,abdominal obesity-metabolic syndrome 1,arteriole,Generalised,Homo sapiens


### 3d. TARKG

In [65]:
TARKG_Disease_Anatomy = pd.read_csv(PROC_DIR + 'TARKG/Disease_Anatomy_TARKG.csv')
TARKG_Disease_Anatomy.columns = TARKG_Disease_Anatomy.columns.str.lower()
print(f"TARKG:    {len(TARKG_Disease_Anatomy):,} rows")

TARKG_Disease_Anatomy['kg_type'] = 'Generalised'
TARKG_Disease_Anatomy['kg_source'] = 'TARKG'
TARKG_Disease_Anatomy['species'] = 'Homo sapiens'

TARKG_Disease_Anatomy.head(2)

TARKG:    7,042 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,head_id_is,tail_id_is,kg_index,kg,change,kg_type,species
0,DOID:3310,Disease_Anatomy,UBERON:0001037,Disease,DlA,Anatomy,TARKG,atopic dermatitis,DOID,UBERON_ID,Hetionet-727317,Hetionet,0,Generalised,Homo sapiens
1,DOID:3310,Disease_Anatomy,UBERON:0001037,Disease,Hetionet::DlA::Disease:Anatomy,Anatomy,TARKG,atopic dermatitis,DOID,UBERON_ID,DRKG-2598519,DRKG,0,Generalised,Homo sapiens


## 4. Check and Fix Duplicate Columns

In [66]:
SOURCE_DFS = [
    ('Monarch_Disease_Anatomy',  Monarch_Disease_Anatomy),
    ('DRKG_Disease_Anatomy',     DRKG_Disease_Anatomy),
    ('Hetionet_Disease_Anatomy', Hetionet_Disease_Anatomy),
    ('TARKG_Disease_Anatomy',    TARKG_Disease_Anatomy),
]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            for i, (_, s) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"  '{col}' occurrence {i} → sample: {s.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicates")

[Monarch_Disease_Anatomy] ✓ no duplicates
[DRKG_Disease_Anatomy] ✓ no duplicates
[Hetionet_Disease_Anatomy] ✓ no duplicates
[TARKG_Disease_Anatomy] ✓ no duplicates


## 5. Align Schemas and Concatenate

In [67]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', ' kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df['kg_type'] = 'Generalised'
final_df.head(2)

Combined: 15,234 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,kg_type
0,C0729734,Disease_AnatomicalEntity,UBERON:0000304,Disease,disease_has_location,AnatomicalEntity,Monarch_KG,MESH,UBERON,Tendon sheath disorder,tendon sheath,NaN,Homo sapiens,Generalised
1,C1263793,Disease_AnatomicalEntity,UBERON:0002411,Disease,disease_has_location,AnatomicalEntity,Monarch_KG,MESH,UBERON,Clitoris neoplasm,clitoris,NaN,Homo sapiens,Generalised


In [68]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


In [69]:
final_df['relation'] = 'Disease_AnatomicalEntity'
final_df['tail_type'] = 'AnatomicalEntity'
final_df

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,kg_type
0,C0729734,Disease_AnatomicalEntity,UBERON:0000304,Disease,disease_has_location,AnatomicalEntity,Monarch_KG,MESH,UBERON,Tendon sheath disorder,tendon sheath,NaN,Homo sapiens,Generalised
1,C1263793,Disease_AnatomicalEntity,UBERON:0002411,Disease,disease_has_location,AnatomicalEntity,Monarch_KG,MESH,UBERON,Clitoris neoplasm,clitoris,NaN,Homo sapiens,Generalised
2,C0334531,Disease_AnatomicalEntity,UBERON:0003074,Disease,disease_has_location,AnatomicalEntity,Monarch_KG,MESH,UBERON,Mesonephric neoplasm,mesonephric duct,NaN,Homo sapiens,Generalised
3,C0343081,Disease_AnatomicalEntity,UBERON:0003532,Disease,disease_has_location,AnatomicalEntity,Monarch_KG,MESH,UBERON,Livedoid vasculopathy,hindlimb skin,NaN,Homo sapiens,Generalised
4,C1290728,Disease_AnatomicalEntity,UBERON:0001684,Disease,disease_has_location,AnatomicalEntity,Monarch_KG,MESH,UBERON,Osteoradionecrosis of the mandible,mandible,NaN,Homo sapiens,Generalised
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15229,DOID:10283,Disease_AnatomicalEntity,UBERON:0002366,Disease,Hetionet::DlA::Disease:Anatomy,AnatomicalEntity,TARKG,DOID,UBERON_ID,prostate cancer,NaN,NaN,Homo sapiens,Generalised
15230,DOID:10941,Disease_AnatomicalEntity,UBERON:0001624,Disease,DlA,AnatomicalEntity,TARKG,DOID,UBERON_ID,intracranial aneurysm,NaN,NaN,Homo sapiens,Generalised
15231,DOID:10941,Disease_AnatomicalEntity,UBERON:0001624,Disease,Hetionet::DlA::Disease:Anatomy,AnatomicalEntity,TARKG,DOID,UBERON_ID,intracranial aneurysm,NaN,NaN,Homo sapiens,Generalised
15232,DOID:3565,Disease_AnatomicalEntity,UBERON:0001624,Disease,DlA,AnatomicalEntity,TARKG,DOID,UBERON_ID,meningioma,NaN,NaN,Homo sapiens,Generalised


In [70]:
final_df['tail_detail_name'] = (
    final_df['tail_detail_name']
    .fillna(final_df['tail'].map(UBERON_dict))
)
final_df['kg_type'] = 'Generalised'
final_df

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,kg_type
0,C0729734,Disease_AnatomicalEntity,UBERON:0000304,Disease,disease_has_location,AnatomicalEntity,Monarch_KG,MESH,UBERON,Tendon sheath disorder,tendon sheath,NaN,Homo sapiens,Generalised
1,C1263793,Disease_AnatomicalEntity,UBERON:0002411,Disease,disease_has_location,AnatomicalEntity,Monarch_KG,MESH,UBERON,Clitoris neoplasm,clitoris,NaN,Homo sapiens,Generalised
2,C0334531,Disease_AnatomicalEntity,UBERON:0003074,Disease,disease_has_location,AnatomicalEntity,Monarch_KG,MESH,UBERON,Mesonephric neoplasm,mesonephric duct,NaN,Homo sapiens,Generalised
3,C0343081,Disease_AnatomicalEntity,UBERON:0003532,Disease,disease_has_location,AnatomicalEntity,Monarch_KG,MESH,UBERON,Livedoid vasculopathy,hindlimb skin,NaN,Homo sapiens,Generalised
4,C1290728,Disease_AnatomicalEntity,UBERON:0001684,Disease,disease_has_location,AnatomicalEntity,Monarch_KG,MESH,UBERON,Osteoradionecrosis of the mandible,mandible,NaN,Homo sapiens,Generalised
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15229,DOID:10283,Disease_AnatomicalEntity,UBERON:0002366,Disease,Hetionet::DlA::Disease:Anatomy,AnatomicalEntity,TARKG,DOID,UBERON_ID,prostate cancer,bulbo-urethral gland,NaN,Homo sapiens,Generalised
15230,DOID:10941,Disease_AnatomicalEntity,UBERON:0001624,Disease,DlA,AnatomicalEntity,TARKG,DOID,UBERON_ID,intracranial aneurysm,anterior cerebral artery,NaN,Homo sapiens,Generalised
15231,DOID:10941,Disease_AnatomicalEntity,UBERON:0001624,Disease,Hetionet::DlA::Disease:Anatomy,AnatomicalEntity,TARKG,DOID,UBERON_ID,intracranial aneurysm,anterior cerebral artery,NaN,Homo sapiens,Generalised
15232,DOID:3565,Disease_AnatomicalEntity,UBERON:0001624,Disease,DlA,AnatomicalEntity,TARKG,DOID,UBERON_ID,meningioma,anterior cerebral artery,NaN,Homo sapiens,Generalised


In [71]:
nan_summary(final_df)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


In [72]:
final_df[final_df['kg_type'].isna()]

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,kg_type


## 6. Standardise Fixed-Value Columns

In [73]:
final_df['tail_id_is'] = 'UBERON'

print("Unique relation:",   set(final_df['relation']))
print("Unique head_type:",  set(final_df['head_type']))
print("Unique tail_type:",  set(final_df['tail_type']))
print("Unique head_id_is:", set(final_df['head_id_is']))
print("Unique tail_id_is:", set(final_df['tail_id_is']))
print("Unique kg_source:",  set(final_df['kg_source']))

Unique relation: {'Disease_AnatomicalEntity'}
Unique head_type: {'Disease'}
Unique tail_type: {'AnatomicalEntity'}
Unique head_id_is: {'MESH', 'DOID'}
Unique tail_id_is: {'UBERON'}
Unique kg_source: {'TARKG', 'Monarch_KG', 'DRKG', 'Hetionet'}


## 7. Deduplicate and Add Schema Columns

In [74]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': merge_sources,
    'species': 'first',
})

final_df_dedup = final_df_dedup[REQUIRED_COLS]

print(f"After deduplication: {len(final_df_dedup):,} rows")
final_df_dedup.head()

After deduplication: 8,826 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,C0001163,Disease_AnatomicalEntity,UBERON:0001648,Disease,disease_has_location,AnatomicalEntity,Monarch_KG,MESH,UBERON,Vestibulocochlear nerve disorder,vestibulocochlear nerve,Generalised,Homo sapiens
1,C0001614,Disease_AnatomicalEntity,UBERON:0001235,Disease,disease_has_location,AnatomicalEntity,Monarch_KG,MESH,UBERON,Adrenal cortex disorder,adrenal cortex,Generalised,Homo sapiens
2,C0002173,Disease_AnatomicalEntity,UBERON:0002073,Disease,disease_has_location,AnatomicalEntity,Monarch_KG,MESH,UBERON,Alopecia mucinosa,hair follicle,Generalised,Homo sapiens
3,C0002631,Disease_AnatomicalEntity,UBERON:0000305,Disease,disease_has_location,AnatomicalEntity,Monarch_KG,MESH,UBERON,Amnionitis,amnion,Generalised,Homo sapiens
4,C0003462,Disease_AnatomicalEntity,UBERON:0001245,Disease,disease_has_location,AnatomicalEntity,Monarch_KG,MESH,UBERON,Anus disorder,anus,Generalised,Homo sapiens


In [75]:
final_df_dedup[final_df_dedup['head_detail_name'].isna()]

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species


## 8. QC — NaN Check

In [76]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


In [77]:
final_df_dedup['head_species'] = 'Homo sapiens'
final_df_dedup['tail_species'] = 'Homo sapiens'


## 9. Save Output

In [78]:
final_df_dedup['head_species'] = 'Homo sapiens'
final_df_dedup['tail_species'] = 'Homo sapiens'


In [79]:
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 8,826 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/DISEASE_ANATOMY/ALL_DISEASE_ANATOMY.csv


# Final Mapping of disease

In [80]:
import pandas as pd

# =========================================================
# LOAD DISEASE MAPPING FILE
# =========================================================

DB_DIR = BASE_DIR + 'data_collection/databases_for_mapping/'

final_file_CH_D = pd.read_csv(
    DB_DIR + 'DO/ultimate_DISEASE_dictionary.csv'
)

# =========================================================
# PRIORITIZE DOID IDs
# =========================================================

# rows having DOID
doid_rows = final_file_CH_D[
    final_file_CH_D['entity_id']
    .str.startswith('DOID:', na=False)
]

# entity names that have at least one DOID
names_with_doid = set(
    doid_rows['entity_name']
    .str.lower()
    .str.strip()
)

# keep:
# 1. all DOID rows
# 2. non-DOID rows only if no DOID exists

final_diseaseid_df = final_file_CH_D[

    final_file_CH_D['entity_id']
    .str.startswith('DOID:', na=False)

    |

    ~final_file_CH_D['entity_name']
    .str.lower()
    .str.strip()
    .isin(names_with_doid)
]

final_diseaseid_df = (
    final_diseaseid_df
    .reset_index(drop=True)
)

# =========================================================
# CREATE DISEASE NAME -> ID DICTIONARY
# =========================================================

disease_dict = dict(

    zip(

        final_diseaseid_df['entity_name']
        .str.lower()
        .str.strip(),

        final_diseaseid_df['entity_id']
    )
)

# =========================================================
# UNIVERSAL ENTITY MAPPING FUNCTION
# =========================================================

def map_entity_ids(
    df,
    mapping_dict,
    side='tail'
):
    """
    Universal KG entity mapper.

    Parameters
    ----------
    df : pd.DataFrame

    mapping_dict : dict
        entity_name -> entity_id

    side : str
        'head' or 'tail'

    Returns
    -------
    pd.DataFrame
    """

    # -----------------------------------------
    # dynamic column names
    # -----------------------------------------

    detail_col = f'{side}_detail_name'

    id_col = side

    id_is_col = f'{side}_id_is'

    # -----------------------------------------
    # map IDs
    # -----------------------------------------

    df[id_col] = (

        df[detail_col]

        .astype(str)

        .str.lower()

        .str.strip()

        .map(mapping_dict)
    )

    # -----------------------------------------
    # assign ID source
    # -----------------------------------------

    df[id_is_col] = None

    # DOID
    # DOID
    df.loc[
        df[id_col].str.startswith('DOID:', na=False),
        id_is_col
    ] = 'DOID'
    
    # everything else -> MESH
    df.loc[
        ~df[id_col].str.startswith('DOID:', na=False)
        &
        df[id_col].notna(),
        id_is_col
    ] = 'MESH'

    return df

In [82]:
OUT_PATH

'/storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/DISEASE_ANATOMY/ALL_DISEASE_ANATOMY.csv'

In [83]:
# =========================================================
# LOAD KG FILE
# =========================================================

final_file = pd.read_csv(OUT_PATH)

# =========================================================
# MAP DISEASE IDS TO TAIL
# =========================================================

final_file = map_entity_ids(df=final_file,mapping_dict=disease_dict,side='head')

# =========================================================
# RESULTS
# =========================================================
final_file

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,head_species,tail_species
0,C0001163,Disease_AnatomicalEntity,UBERON:0001648,Disease,disease_has_location,AnatomicalEntity,Monarch_KG,MESH,UBERON_ID,Vestibulocochlear nerve disorder,vestibulocochlear nerve,Generalised,Homo sapiens,Homo sapiens,Homo sapiens
1,C0001614,Disease_AnatomicalEntity,UBERON:0001235,Disease,disease_has_location,AnatomicalEntity,Monarch_KG,MESH,UBERON_ID,Adrenal cortex disorder,adrenal cortex,Generalised,Homo sapiens,Homo sapiens,Homo sapiens
2,C0002173,Disease_AnatomicalEntity,UBERON:0002073,Disease,disease_has_location,AnatomicalEntity,Monarch_KG,MESH,UBERON_ID,Alopecia mucinosa,hair follicle,Generalised,Homo sapiens,Homo sapiens,Homo sapiens
3,C0002631,Disease_AnatomicalEntity,UBERON:0000305,Disease,disease_has_location,AnatomicalEntity,Monarch_KG,MESH,UBERON_ID,Amnionitis,amnion,Generalised,Homo sapiens,Homo sapiens,Homo sapiens
4,C0003462,Disease_AnatomicalEntity,UBERON:0001245,Disease,disease_has_location,AnatomicalEntity,Monarch_KG,MESH,UBERON_ID,Anus disorder,anus,Generalised,Homo sapiens,Homo sapiens,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8821,DOID:9970,Disease_AnatomicalEntity,UBERON:0002385,Disease,Disease–localizes–Anatomy,AnatomicalEntity,Hetionet::TARKG,DOID,UBERON_ID,obesity,muscle tissue,Generalised,Homo sapiens,Homo sapiens,Homo sapiens
8822,DOID:9970,Disease_AnatomicalEntity,UBERON:0002410,Disease,Disease–localizes–Anatomy,AnatomicalEntity,Hetionet::TARKG,DOID,UBERON_ID,obesity,autonomic nervous system,Generalised,Homo sapiens,Homo sapiens,Homo sapiens
8823,DOID:9970,Disease_AnatomicalEntity,UBERON:0002522,Disease,Disease–localizes–Anatomy,AnatomicalEntity,Hetionet::TARKG,DOID,UBERON_ID,obesity,tunica media,Generalised,Homo sapiens,Homo sapiens,Homo sapiens
8824,DOID:9970,Disease_AnatomicalEntity,UBERON:0002523,Disease,Disease–localizes–Anatomy,AnatomicalEntity,Hetionet::TARKG,DOID,UBERON_ID,obesity,tunica intima,Generalised,Homo sapiens,Homo sapiens,Homo sapiens


In [85]:
final_file[final_file['head'].isna()]

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,head_species,tail_species


In [86]:
final_file.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_file):,} rows → {OUT_PATH}")

Saved 8,826 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/DISEASE_ANATOMY/ALL_DISEASE_ANATOMY.csv
